# Preprocessing

This notebook builds missing lemma-count matrices and validates all matrices against the fixed selection.
Download the IMDb62 source archive as described in the README before running it.
Set REBUILD to True to recompute all three corpora from their original archives.

## Setup

In [ ]:
from collections import Counter
from hashlib import file_digest
from io import TextIOWrapper
from pathlib import Path
import re
import unicodedata
import xml.etree.ElementTree as ET
from zipfile import ZipFile

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction import DictVectorizer

ROOT = Path.cwd()
REBUILD = False
CORPORA = ("IMDb62", "FreDraCor", "RusLit9")
MODELS = {"IMDb62": "en_core_web_sm", "FreDraCor": "fr_core_news_sm", "RusLit9": "ru_core_news_sm"}
RELEASES = {"IMDb62": "imdb62.zip", "FreDraCor": "FreDraCor.zip", "RusLit9": "RusLit_4509680.zip"}
# The selection fixes the corpus rows and their order.
selection = pd.read_csv(ROOT / "data/selection.csv.gz", dtype=str, keep_default_na=False)

# Restore IMDb62 IDs locally from the exact release, in its original row order.
imdb_rows = selection.corpus.eq("IMDb62")
if selection.loc[imdb_rows, "text_id"].eq("").all():
    imdb_path = ROOT / "data/raw" / RELEASES["IMDb62"]
    if not imdb_path.is_file():
        raise FileNotFoundError("Download data/raw/imdb62.zip using the README instructions.")
    with imdb_path.open("rb") as source:
        digest = file_digest(source, "sha256").hexdigest()
    if digest != "88091d873ca90876304b3f11dd8bf27547fe238ef84ab526f065eedda5fb3d4e":
        raise ValueError("The IMDb62 archive differs from the release specified in README.md.")
    with ZipFile(imdb_path) as archive, TextIOWrapper(
        archive.open("imdb62.txt"), encoding="utf-8-sig"
    ) as reviews:
        ids_and_authors = [line.split("\t", 2)[:2] for line in reviews]
    selection.loc[imdb_rows, "text_id"] = [f"IMDb62:{review}" for review, _ in ids_and_authors]
    selection.loc[imdb_rows, "author"] = [author for _, author in ids_and_authors]

# Keep words made of letters, including internal apostrophes.
WORD = re.compile(r"[^\W\d_]+(?:'[^\W\d_]+)*", re.UNICODE)
NS = {"t": "http://www.tei-c.org/ns/1.0"}
NAMES = dict(Bryusov="Валерий Яковлевич Брюсов", Chekhov="Антон Павлович Чехов",
             Dostoevsky="Федор Михайлович Достоевский", Gogol="Николай Васильевич Гоголь",
             Gorky="Максим Горький", Herzen="Александр Иванович Герцен",
             Pushkin="Александр Сергеевич Пушкин",
             Tolstoy="Лев Николаевич Толстой", Turgenev="Иван Сергеевич Тургенев")

## Extract whole texts
IMDb62 uses review title plus content. FreDraCor uses spoken TEI text, omitting
speaker names, stage directions, notes and page/column markers. RusLit9 removes
exact author-name bylines near the start and joins the four War and Peace volumes.

`data/selection.csv.gz` fixes the original study's rows, author labels and evaluation
groups. It contains no text-derived feature values. The RusLit `prose` selection also
includes some plays and essays. One matrix row remains one complete source observation.

In [ ]:
def speech(root):
    banned = {"speaker", "stage", "note", "fw", "pb", "cb"}

    def inner(node):
        if node.tag.split("}")[-1] in banned:
            return ""
        return (node.text or "") + "".join(inner(child) + (child.tail or "") for child in node)

    body = root.find("./t:text/t:body", NS)
    if body is None:
        return ""
    return "\n".join(inner(child) for sp in body.findall(".//t:sp", NS)
                     for child in sp if child.tag.split("}")[-1] not in banned)

def clean_byline(text, author):
    lines = text.splitlines(keepends=True)
    full = NAMES[author].lower().replace("ё", "е")
    parts = full.split()
    accepted = {full, " ".join((parts[0], parts[-1])), parts[-1]}
    for index in range(min(20, len(lines))):
        value = re.sub(r"[^\w\s]", "", lines[index].lower().replace("ё", "е"))
        if " ".join(value.split()) in accepted:
            lines[index] = ""
    return "".join(lines)

def blocks(text):
    """Bound NLP memory without changing the whole-text observation unit."""
    start = 0
    while start < len(text):
        stop = min(start + 20000, len(text))
        if stop < len(text):
            while stop > start and not text[stop - 1].isspace():
                stop -= 1
            if stop == start:
                raise ValueError("A whitespace-free token exceeds 20000 characters")
        yield text[start:stop]
        start = stop

In [ ]:
def extract_texts(corpus, rows):
    with ZipFile(ROOT / "data/raw" / RELEASES[corpus]) as archive:
        # Reviews combine title and content; plays retain spoken text only.
        if corpus == "IMDb62":
            reviews = {}
            for line in archive.read("imdb62.txt").decode("utf-8-sig").splitlines():
                identity, author, _, _, title, content = line.split("\t")
                reviews["IMDb62:" + identity] = (author, title + " " + content)
        if corpus == "RusLit9":
            members = {name.split("/", 1)[1]: name for name in archive.namelist() if "/" in name}
        for row in rows:
            if corpus == "IMDb62":
                author, text = reviews[row["text_id"]]
                assert author == row["author"]
            elif corpus == "FreDraCor":
                text = speech(ET.fromstring(archive.read(row["source_member"])))
            else:
                parts = []
                for name in row["source_member"].split(";"):
                    text = archive.read(members[name]).decode("utf-8-sig")
                    parts.append(clean_byline(text, row["author"]))
                text = "\n\n".join(parts)
            yield text

## Lemmatize and count
Use spaCy 3.8.11 with language models 3.8.0. Lowercase and NFC-normalize lemmas,
normalize apostrophes, and retain Unicode-letter tokens with optional internal
apostrophes. No stopword removal or feature selection occurs here.
Long texts use whitespace-bounded NLP blocks of at most 20,000 characters, reassembled
into a single observation. Relative frequencies are computed from counts in notebook 2.

In [ ]:
def preprocess(corpus):
    import spacy

    rows = selection.loc[selection.corpus.eq(corpus)].to_dict("records")
    model = spacy.load(MODELS[corpus], exclude=["parser", "ner", "senter"])
    lengths = []

    # Count lemmas for each complete text, even when it needs multiple blocks.
    def dictionaries():
        for index, text in enumerate(extract_texts(corpus, rows)):
            counts = Counter()
            for document in model.pipe(blocks(text), batch_size=16, n_process=1):
                for token in document:
                    if token.is_space or token.is_punct:
                        continue
                    if not token.lemma_:
                        raise ValueError("Missing lemma annotation")
                    lemma = token.lemma_.replace("\u2019", "'").replace("\u2018", "'").lower()
                    lemma = unicodedata.normalize("NFC", lemma)
                    counts.update(WORD.findall(lemma))
            length = sum(counts.values())
            if length == 0:
                raise ValueError(f"No retained lemmas: {rows[index]['text_id']}")
            lengths.append(length)
            if (index + 1) % (1000 if corpus == "IMDb62" else 50) == 0:
                print(f"{corpus}: {index + 1}/{len(rows)} texts", flush=True)
            yield counts

    # Each row of the sparse matrix is one text; each column is one lemma.
    vectorizer = DictVectorizer(dtype=np.int32, sort=True)
    counts = vectorizer.fit_transform(dictionaries()).tocsr()
    authors, labels = np.unique([row["author"] for row in rows], return_inverse=True)
    folder = ROOT / "data/preprocessed" / corpus
    folder.mkdir(parents=True, exist_ok=True)
    sparse.save_npz(folder / "counts.npz", counts)
    np.savez_compressed(folder / "metadata.npz", labels=labels, authors=authors,
                        words=vectorizer.get_feature_names_out().astype(str),
                        text_ids=np.array([row["text_id"] for row in rows]),
                        groups=np.array([row["group"] for row in rows]),
                        token_count=np.array(lengths))
    print(
        f"Saved {corpus}: {counts.shape[0]:,} whole texts, "
        f"{len(authors)} authors, {counts.shape[1]:,} lemmas."
    )

## Build and verify the matrices

Missing matrices are built from the original archives. Set REBUILD to True to rebuild existing matrices.

In [ ]:
expected = {"IMDb62": (62000, 62), "FreDraCor": (1191, 87), "RusLit9": (259, 9)}
for corpus in CORPORA:
    folder = ROOT / "data/preprocessed" / corpus
    if REBUILD or not ((folder / "counts.npz").is_file() and (folder / "metadata.npz").is_file()):
        preprocess(corpus)
    rows = selection.loc[selection.corpus.eq(corpus)]
    counts = sparse.load_npz(folder / "counts.npz")
    with np.load(folder / "metadata.npz", allow_pickle=False) as meta:
        assert (counts.shape[0], len(meta["authors"])) == expected[corpus]
        np.testing.assert_array_equal(meta["text_ids"], rows.text_id.to_numpy())
        np.testing.assert_array_equal(meta["groups"], rows.group.to_numpy())
        np.testing.assert_array_equal(meta["authors"][meta["labels"]], rows.author.to_numpy())
        np.testing.assert_array_equal(np.asarray(counts.sum(axis=1)).ravel(), meta["token_count"])
        assert counts.shape[1] == len(meta["words"]) and (meta["token_count"] > 0).all()
        assert rows.groupby("group").author.nunique().eq(1).all()
        if corpus == "RusLit9":
            assert len(np.unique(meta["groups"])) == 254
    print(f"Verified {corpus}: {counts.shape[0]:,} texts, {counts.shape[1]:,} lemmas.")


Continue with [02_experiments.ipynb](02_experiments.ipynb).